# OP26 — Phase 2: Exploratory Data Analysis
Reads the Phase-1 outputs and produces deck-ready figures, the **empirical peak-window definition**, EDA summary tables, and a data-driven findings doc. Logic lives in `eda.py`; run top-to-bottom to regenerate everything in `figures/` and the `eda_*` files in `outputs/`.

**Two band concepts are kept distinct:** *operational* bands (per zone-hour: util ≥ 0.80 surge, < 0.30 discount — the brief's triggers) vs *temporal* peak windows (which hours are systematically busy — defined here by tertiles of mean hourly utilization).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from IPython.display import Image, display
import config as C, eda as E
panel, zf, info, acn = E.load()
print('panel', panel.shape, '| zones', len(zf), '| acn', len(acn))

## Temporal profiling + empirical peak windows
Occupancy peaks **overnight** (vehicles sit plugged in) and troughs **midday**; weekday and weekend are near-identical, so the cycle is time-of-day, not day-of-week. The midday trough is the prime discount window; network-wide surge is essentially absent (it lives in a zone tail).

In [ ]:
overall, wd, we = E.temporal(panel)
E.fig_intraday(overall, wd, we); E.fig_heatmap(panel)
display(pd.read_csv(C.OUTPUT_DIR/'peak_windows.csv').query("scope=='overall'")[['hour','mean_util','band']])
display(Image(C.FIG_DIR/'fig01_intraday_utilization.png'))
display(Image(C.FIG_DIR/'fig02_util_heatmap_hour_dow.png'))

## Volatility by band
The quiet midday (off-peak) window is modestly but consistently the least predictable → daytime discounts should be scheduled/sustained, not reactive.

In [ ]:
vol = E.fig_volatility(panel, overall)
display(vol)
display(Image(C.FIG_DIR/'fig03_volatility_by_band.png'))

## Spatial profile — and the CBD surprise
Congestion is concentrated in the **top ~10% of zones (~all surge incidence)**, and those are **not** the CBD (CBD util is actually lower). → surge must be targeted by *observed* utilization, not the CBD label.

In [ ]:
cbd = E.fig_zone_dist_cbd(zf); E.fig_map(zf, info); E.fig_top_bottom(zf)
display(cbd)
for f in ['fig04_zone_utilization_distribution','fig05_spatial_map','fig06_top_bottom_zones']:
    display(Image(C.FIG_DIR/f'{f}.png'))

## Price vs demand — preliminary (associational) elasticity
Across the 108 price-varying zones, higher tariffs associate with lower demand. Reported as an **associational** signal (per the brief's no-causal-claims rule); re-estimated rigorously in Phase 4.

In [ ]:
elasticity, n_pv = E.fig_price_demand(panel)
print(f'preliminary elasticity ~ {elasticity:+.2f} over {n_pv} price-varying zones')
display(Image(C.FIG_DIR/'fig07_price_vs_demand.png'))

## ACN behaviour
Long post-charge idle (connector blocking), a strong morning arrival peak, and mostly anonymous sessions (only ~15% identified). → idle/occupancy fees + a morning surge can free capacity.

In [ ]:
acn_sum = E.fig_acn(acn)
display(pd.DataFrame([acn_sum]).T.rename(columns={0:'value'}))
display(Image(C.FIG_DIR/'fig08_acn_behaviour.png'))

## Write the findings doc

In [ ]:
E.write_findings(panel, overall, vol, cbd, elasticity, n_pv, acn_sum)
print((C.OUTPUT_DIR/'eda_findings.md').read_text()[:1500])

---
**Phase 2 complete.** `peak_windows.csv` + the EDA findings feed Phase 3 (demand agent) and Phase 4 (tariff agent).